In [27]:
import sys
import numpy as np
import pandas as pd

from typing import Literal
from collections import Counter
from ase import Atoms
from ase.neighborlist import natural_cutoffs
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core.surface import Slab, Structure
from pymatgen.core import Composition
from pymatgen.core.surface import (
    Slab, get_symmetrically_distinct_miller_indices, SlabGenerator)
from ase.neighborlist import NeighborList, natural_cutoffs
from ase.io import read, Trajectory

from gnome_dreams_oer_screening.oer.oer_study_utils import (
    eval_stoichiometry_diff, get_stoichiometry_diff,
    get_stoichiometry_dict)
from gnome_dreams_oer_screening import OER_catalyst_study

ZrO2_m = read('ZrO2_m.cif')
TiO2_r = read('TiO2_rutile.cif')
test_cands = Trajectory('test_cands.traj')

atoms = test_cands[18]
S = AseAtomsAdaptor.get_structure(atoms)

In [28]:
# view(ZrO2_m)

In [29]:
cand =  test_cands[18] # TiO2_r
cand

Atoms(symbols='ZnBi3IrRh3O14', pbc=True, cell=[[6.359742925812065, 0.0, 3.677567832273403], [2.11886857481291, 5.771485047280466, 3.6728729851736506], [0.0, 0.0, 7.34648452]])

In [30]:
study = OER_catalyst_study(TiO2_r,     
    H2O_gas_free_energy = -14.217,
    H2_gas_free_energy = -6.77,
    use_init_as_relaxed_bulk=True
)
study.bulk_atoms

Atoms(symbols='Ti2O4', pbc=True, cell=[4.59983732, 4.59983732, 2.95921356], spacegroup_kinds=...)

In [31]:
study.identify_distinct_surfaces(max_miller=1)
study.get_distinct_millers()

[(1, 1, 1), (1, 1, 0), (1, 0, 1), (1, 0, 0), (0, 0, 1)]

In [41]:
O_indecies = []
for atom in TiO2_r:
    if atom.symbol == 'O':
        O_indecies.append(atom.index)
print(O_indecies)

[2, 3, 4, 5]


In [ ]:
natural_cutoffs(TiO2_r)
nl = NeighborList(natural_cutoffs(TiO2_r), self_interaction=False, 
                  bothways=True, skin=1e-5)

nl.update(TiO2_r)
indices, offsets = nl.get_neighbors(0)
[i for i in indices if i not in O_indecies]

[np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1)]

In [32]:
study.predict_most_likely_surfaces()
study.termination_ranking_df.sort_values('Score')
terminations = study.terminations

In [33]:
df = study.get_termination_rankings().sort_values('Score', ascending=False)
df

,Miller indices,Termination index,Score,Normalized score
2,"(1, 1, 0)",2,-0.103895,1.000000
4,"(1, 0, 0)",4,-0.146930,0.760490
3,"(1, 0, 1)",3,-0.238485,0.250942
0,"(1, 1, 1)",0,-0.279671,0.021723
1,"(1, 1, 1)",1,-0.279671,0.021723
5,"(0, 0, 1)",5,-0.283574,0.000000


In [47]:
atoms_to_view = []
for i, row in df.iterrows():
    slab = study.terminations[row['Termination index']]
    atoms_to_view.append(AseAtomsAdaptor.get_atoms(slab))

view(atoms_to_view)

<Popen: returncode: None args: ['c:\\Users\\matnis\\OneDrive - Danmarks Tekn...>

In [17]:
2 in study.oer_surface_studies

False

In [18]:


slabs = study._build_surfaces((1,0,1), stoichiometry_tolerance = 0.2,
    all_species_present = True,
    select_closest_O_stoichiometry = True,
    max_atoms=1000, max_slab_thickness=1000,
    ignore_stoichiometry_of_species = [])

atoms_list = []
for slab in slabs:
    print(study.compute_slab_thickness(slab))
    atoms_list.append(AseAtomsAdaptor.get_atoms(slab))

        
view(atoms_list)

AttributeError: 'OER_catalyst_study' object has no attribute 'compute_slab_thickness'

In [ ]:
for slab in slabs:
    print(len(slab))

48


In [ ]:
for slab in slabs:
    d = get_stoichiometry_diff(
        get_stoichiometry_dict(study.bulk_structure),
        get_stoichiometry_dict(slab)) 
    print(d['O'])
    print( eval_stoichiometry_diff(study.bulk_structure, slab,
                                max_diff_dict={'O': 0.02},   
                                )
           )
    print('_____________________________________')


0.0
True
_____________________________________
